# nb_10_bronze_ingest

Reads the raw regional CSV files from `Files/landing/raw` and stores them as Delta tables, without changing the business data.

What this notebook does:
1. Finds the files that have not been loaded yet (safe to run again, it never loads a file twice)
2. Reads each group of files that share the same columns, so a column added mid-year does not shift the data
3. Adds audit columns to every row: `_batch_id`, `_load_ts`, `_source_file`, `_region`, `_dt`
4. Appends to `bronze_*` tables. New columns are added to the table automatically
5. Compares rows loaded with the row count promised in the manifest
6. Writes what happened to the control tables `ctl_run_log` and `ctl_file_log`

Input: `Files/landing/raw/<region>/dt=<date>/<table>.csv`, `Files/landing/raw/reference/*.csv`, `Files/landing/raw/_manifest.csv`

Output: `bronze_crm_client`, `bronze_pos_transaction`, `bronze_ecom_order`, `bronze_after_sales_ticket`, `bronze_appointment`, `bronze_product`, `bronze_boutique`, `bronze_fx_rate`, `bronze_manifest`, `ctl_run_log`, `ctl_file_log`


In [ ]:
# ---- Settings (change LOAD_UP_TO to load the data in stages) ----
import os

LOAD_UP_TO = None        # To load every file -> "LOAD_UP_TO = None"
LOCAL_ROOT = os.environ.get("MM_LOCAL_ROOT", "/lakehouse/default/Files/landing/raw")   # Python lists files
SPARK_ROOT = os.environ.get("MM_SPARK_ROOT", "Files/landing/raw")                      # Spark reads files

REGIONS = ["americas", "europe", "asia_pacific"]
REGIONAL_TABLES = ["crm_client", "pos_transaction", "ecom_order", "after_sales_ticket", "appointment"]
REFERENCE_TABLES = ["product", "boutique", "fx_rate"]

In [ ]:
# ---- Setup ----
import re
import uuid
from datetime import datetime, timezone

import pyspark.sql.functions as F
from pyspark.sql import Row

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
BATCH_START = datetime.now(timezone.utc).replace(tzinfo=None)
print("batch id:", BATCH_ID)

run_log = []          # one row per object loaded, written to ctl_run_log at the end
file_log = []         # one row per file checked, written to ctl_file_log at the end
failures = []         # collected so that even if notebook fails at the end after logging everything


def table_exists(name):
    return spark.catalog.tableExists(name)


def log_run(obj, status, files, rows, started, failure_class=None, message=None):
    run_log.append(Row(
        batch_id=BATCH_ID, layer="bronze", object_name=obj, status=status,
        files_loaded=int(files), rows_loaded=int(rows),
        started_at=started, finished_at=datetime.now(timezone.utc).replace(tzinfo=None),
        failure_class=failure_class, message=message))

In [ ]:
# ---- Step 1: list the files in the landing area ----
pattern = re.compile(r"^(?P<region>[a-z_]+)/dt=(?P<dt>\d{4}-\d{2}-\d{2})/(?P<table>[a-z_]+)\.csv$")

landed = []   # one dict per regional file
for region in REGIONS:
    region_dir = os.path.join(LOCAL_ROOT, region)
    if not os.path.isdir(region_dir):
        continue
    for day_dir in sorted(os.listdir(region_dir)):
        day_path = os.path.join(region_dir, day_dir)
        if not os.path.isdir(day_path):
            continue
        for fname in sorted(os.listdir(day_path)):
            rel = f"{region}/{day_dir}/{fname}"
            m = pattern.match(rel)
            if not m or m.group("table") not in REGIONAL_TABLES:
                continue
            if LOAD_UP_TO and m.group("dt") > LOAD_UP_TO:
                continue
            with open(os.path.join(day_path, fname), "r", encoding="utf-8") as fh:
                header = fh.readline().strip()
            landed.append(dict(rel=rel, region=region, dt=m.group("dt"),
                               table=m.group("table"), header=header))

print("regional files in the landing area:", len(landed))

In [ ]:
# ---- Step 2: load the regional tables ----
def load_regional_table(table):
    started = datetime.now(timezone.utc).replace(tzinfo=None)
    target = f"bronze_{table}"
    try:
        files = [f for f in landed if f["table"] == table]

        # loaded => file already in the bronze table
        already = set()
        if table_exists(target):
            already = {r["_source_file"] for r in spark.table(target).select("_source_file").distinct().collect()}
        new_files = [f for f in files if f["rel"] not in already]
        if not new_files:
            log_run(target, "skipped_no_new_files", 0, 0, started)
            print(f"{target}: nothing new")
            return

        # Files with different column lists are read separately, then stacked by column name
        by_header = {}
        for f in new_files:
            by_header.setdefault(f["header"], []).append(f)

        parts = []
        for header, group in by_header.items():
            paths = [f"{SPARK_ROOT}/{f['rel']}" for f in group]
            part = (spark.read.option("header", True).option("inferSchema", False)
                    .option("mode", "PERMISSIVE").csv(paths))      # every column to stay text in bronze
            part = part.withColumn("_source_file",
                                   F.regexp_extract(F.col("_metadata.file_path"), r"landing/raw/(.*)$", 1))
            parts.append(part)

        df = parts[0]
        for p in parts[1:]:
            df = df.unionByName(p, allowMissingColumns=True)

        df = (df
              .withColumn("_region", F.regexp_extract(F.col("_source_file"), r"^([a-z_]+)/", 1))
              .withColumn("_dt", F.regexp_extract(F.col("_source_file"), r"dt=(\d{4}-\d{2}-\d{2})", 1))
              .withColumn("_batch_id", F.lit(BATCH_ID))
              .withColumn("_load_ts", F.current_timestamp()))

        # If the table already has columns this batch lacks (an older layout arriving late),
        # then its filled with empty values so the append lines up
        if table_exists(target):
            for c in spark.table(target).columns:
                if c not in df.columns:
                    df = df.withColumn(c, F.lit(None).cast("string"))

        # Rows per file, kept for the reconciliation step
        per_file = {r["_source_file"]: r["n"] for r in df.groupBy("_source_file").agg(F.count("*").alias("n")).collect()}

        (df.write.format("delta").mode("append")
           .option("mergeSchema", "true")                 # a new source column is added to the table
           .saveAsTable(target))

        rows = sum(per_file.values())
        loaded_files[table] = per_file
        log_run(target, "success", len(new_files), rows, started)
        print(f"{target}: {len(new_files)} files, {rows} rows, {len(by_header)} column layout(s)")
    except Exception as e:  # noqa: BLE001
        log_run(target, "failed", 0, 0, started, failure_class="processing", message=str(e)[:500])
        failures.append((target, e))
        print(f"{target}: FAILED -> {e}")


loaded_files = {}
for t in REGIONAL_TABLES:
    load_regional_table(t)

In [ ]:
# ---- Step 3: reference tables and manifest (full copies, replaced on every run) ----
def save_reference(df, path, target, started):
    df = (df.withColumn("_source_file", F.lit(path))
            .withColumn("_batch_id", F.lit(BATCH_ID))
            .withColumn("_load_ts", F.current_timestamp()))
    n = df.count()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    log_run(target, "success", 1, n, started)
    print(f"{target}: {n} rows")


def load_reference(path, target):
    started = datetime.now(timezone.utc).replace(tzinfo=None)
    try:
        if os.path.basename(path).startswith("_"):
            # manifest is read with Python's csv module becaise Spark skips files whose names start with "_" 
            import csv
            with open(os.path.join(LOCAL_ROOT, path), newline="", encoding="utf-8") as fh:
                reader = csv.reader(fh)
                cols = next(reader)
                df = spark.createDataFrame([tuple(r) for r in reader], schema=cols)   # all text
        else:
            df = spark.read.option("header", True).option("inferSchema", False).csv(f"{SPARK_ROOT}/{path}")
        save_reference(df, path, target, started)
    except Exception as e:  # noqa: BLE001
        log_run(target, "failed", 0, 0, started, failure_class="processing", message=str(e)[:500])
        failures.append((target, e))
        print(f"{target}: FAILED -> {e}")


for t in REFERENCE_TABLES:
    load_reference(f"reference/{t}.csv", f"bronze_{t}")
load_reference("_manifest.csv", "bronze_manifest")

In [ ]:
# ---- Step 4: reconcile with the manifest ----
# The manifest says how many rows each source promised to send and when the file should arrive
# For every file checked in this run, record: ok, short_file (fewer rows than promised),
# missing_file (promised but not in the landing area) or late (arrived more than 20 hours after expected)
if table_exists("bronze_manifest"):
    manifest = [r.asDict() for r in spark.table("bronze_manifest").collect()]
    if LOAD_UP_TO:
        manifest = [m for m in manifest if m["dt"] <= LOAD_UP_TO]

    present = {f["rel"] for f in landed}
    rows_by_file = {}
    for per_file in loaded_files.values():
        rows_by_file.update(per_file)

    for m in manifest:
        rel = m["file_path"]
        promised = int(m["row_count"])
        lag_hours = (datetime.fromisoformat(m["actual_arrival_ts"]) -
                     datetime.fromisoformat(m["expected_arrival_ts"])).total_seconds() / 3600
        if rel not in present:
            status, actual = "missing_file", None
        elif rel not in rows_by_file:
            continue                                  # loaded in an earlier batch, already checked then
        else:
            actual = int(rows_by_file[rel])
            status = "short_file" if actual < promised else ("late" if lag_hours > 20 else "ok")
        file_log.append(Row(
            batch_id=BATCH_ID, source_file=rel, region=m["region"], table_name=m["table"], dt=m["dt"],
            rows_promised=promised, rows_loaded=actual, arrival_lag_hours=round(lag_hours, 2),
            status=status, failure_class=("source" if status in ("short_file", "missing_file", "late") else None)))

    summary = {}
    for r in file_log:
        summary[r.status] = summary.get(r.status, 0) + 1
    print("file checks:", summary)

In [ ]:
# ---- Step 5: write the control tables ----
# The column types are written out because a column that is empty in every row (for example
# failure_class when nothing failed) cannot be guessed by Spark
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType,
                               DoubleType, TimestampType)

RUN_LOG_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("object_name", StringType()), StructField("status", StringType()),
    StructField("files_loaded", IntegerType()), StructField("rows_loaded", LongType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("failure_class", StringType()), StructField("message", StringType())])

FILE_LOG_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("source_file", StringType()),
    StructField("region", StringType()), StructField("table_name", StringType()),
    StructField("dt", StringType()), StructField("rows_promised", LongType()),
    StructField("rows_loaded", LongType()), StructField("arrival_lag_hours", DoubleType()),
    StructField("status", StringType()), StructField("failure_class", StringType())])

if run_log:
    (spark.createDataFrame(run_log, schema=RUN_LOG_SCHEMA).write.format("delta").mode("append")
          .option("mergeSchema", "true").saveAsTable("ctl_run_log"))
if file_log:
    (spark.createDataFrame(file_log, schema=FILE_LOG_SCHEMA).write.format("delta").mode("append")
          .option("mergeSchema", "true").saveAsTable("ctl_file_log"))

if failures:
    names = ", ".join(n for n, _ in failures)
    raise RuntimeError(f"Bronze load finished with failures in: {names}. See ctl_run_log batch {BATCH_ID}.")
print("bronze load finished, batch", BATCH_ID)

## Result

In [ ]:
# Rows per bronze table and how many batches loaded them
for t in ["crm_client", "pos_transaction", "ecom_order", "after_sales_ticket", "appointment",
          "product", "boutique", "fx_rate", "manifest"]:
    name = f"bronze_{t}"
    if table_exists(name):
        d = spark.table(name)
        print(f"{name:28s} rows={d.count():>8}  batches={d.select('_batch_id').distinct().count()}  columns={len(d.columns)}")

In [ ]:
# Schema evolution: the new columns exist in the table, and are empty before the day they appeared
if table_exists("bronze_crm_client"):
    d = spark.table("bronze_crm_client")
    if "client_segment" in d.columns:
        d.filter("_region = 'europe'").groupBy("_dt").agg(F.count("*").alias("rows"),
            F.count("client_segment").alias("with_segment")).orderBy("_dt").show(40, truncate=False)

if table_exists("bronze_pos_transaction"):
    d = spark.table("bronze_pos_transaction")
    if "sales_associate_id" in d.columns:
        d.filter("_region = 'americas'").groupBy(F.substring("_dt", 1, 7).alias("month")).agg(
            F.count("*").alias("rows"), F.count("sales_associate_id").alias("with_associate")
        ).orderBy("month").show(20, truncate=False)

In [ ]:
# Problems found when comparing with the manifest
if table_exists("ctl_file_log"):
    spark.table("ctl_file_log").filter("status != 'ok'").groupBy("batch_id", "status").count().orderBy("batch_id", "status").show(truncate=False)
if table_exists("ctl_run_log"):
    spark.table("ctl_run_log").orderBy(F.col("started_at").desc()).show(12, truncate=False)